# 06 · Explainable AI (as V3)

Both methods use the **held-out** outer test folds of the main runs (106 m) and the fold models saved in notebook 03: each fold's model explains only its own test boreholes.

| Method | What it measures |
|---|---|
| Permutation importance | drop in balanced accuracy when an input is shuffled (10 shuffles), also for **location** (Easting + Northing together) and **elevation** |
| SHAP (exact Shapley values) | how much each input raises or lowers P(Clay) for 300 test cells per fold; background = 100 training cells of that fold |

**Run time:** 20-60 minutes (the GW models are the slowest).

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib, warnings
import shap
from sklearn.metrics import balanced_accuracy_score

cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
dev = cells[cells.set == "development"]
preds = pd.read_csv(H.out_dir(S, "03_spatial_cv", private=True) / "outer_predictions.csv")
main_block = H.block_sizes(S)[0]
folds = H.load_folds(S, main_block)
MODELS_DIR = H.out_dir(S, "03_spatial_cv", private=True) / "models"
OUT, PRIV = H.out_dir(S, "06_xai"), H.out_dir(S, "06_xai", private=True)
models = [m for m in S["cv"]["models"] if m in set(preds.model)]
folds_k = sorted(preds[(preds.block_m == main_block) & (preds.purge_m == 0)].outer_fold.unique())
NAMES = {"UTM_Easting_m": "Easting", "UTM_Northing_m": "Northing", "Top_Elevation_m": "Elevation"}
GROUPS = {"Location (Easting + Northing)": ["UTM_Easting_m", "UTM_Northing_m"], "Elevation (depth trend)": ["Top_Elevation_m"]}
REPEATS = 3 if S["run"]["quick"] else 10

def load_fold(model, k):
    m = joblib.load(MODELS_DIR / f"A_{main_block}m_p0_{model}_fold{k}.joblib")
    thr = float(preds[(preds.model == model) & (preds.block_m == main_block) & (preds.purge_m == 0)
                      & (preds.outer_fold == k)].threshold.iloc[0])
    train, test = H.outer_split(dev, folds, k)
    return m, thr, train, test

## 1. Permutation importance (held-out)

In [ ]:
rng = np.random.default_rng(S["run"]["seed"])
rows = []
for model in models:
    for k in folds_k:
        m, thr, _, test = load_fold(model, k)
        X0, y = test[H.FEATURES].values, test.y.values
        meas = test.measured.values == 1

        def score(X, mask):
            p = m.predict_proba(X)[:, 1]
            return balanced_accuracy_score(y[mask], (p[mask] >= thr).astype(int))

        for subset, mask in (("all cells", np.ones(len(y), bool)), ("measured only", meas)):
            base = score(X0, mask)
            for name, cols in [*[(NAMES[f], [f]) for f in H.FEATURES], *GROUPS.items()]:
                idx = [H.FEATURES.index(c) for c in cols]
                drops = []
                for _ in range(REPEATS):
                    X = X0.copy()
                    order = rng.permutation(len(X))
                    X[:, idx] = X0[order][:, idx]          # the inputs of a group are shuffled together
                    drops.append(base - score(X, mask))
                rows.append({"model": model, "fold": k, "cells": subset, "input": name, "kind": "",
                             "baseline_bal_acc": base, "drop_mean": np.mean(drops)})
    print(model, "done", flush=True)
perm = pd.DataFrame(rows)
perm["kind"] = np.where(perm.input.isin(GROUPS), "group", "single")
perm.to_csv(OUT / "permutation_importance_by_fold.csv", index=False)
perm_sum = perm.groupby(["cells", "kind", "model", "input"]).drop_mean.agg(["mean", "std"]).round(4).reset_index()
perm_sum.to_csv(OUT / "permutation_importance_summary.csv", index=False)
display(perm_sum[perm_sum.cells == "all cells"].pivot_table(index=["kind", "input"], columns="model", values="mean").round(3))

## 2. SHAP values (held-out cells)

In [ ]:
N_EXPLAIN = 60 if S["run"]["quick"] else 300
N_BACKGROUND = 50 if S["run"]["quick"] else 100
parts = []
for model in models:
    for k in folds_k:
        m, thr, train, test = load_fold(model, k)
        r = np.random.default_rng(seed=[S["run"]["seed"], k])
        bg = train[H.FEATURES].values[r.choice(len(train), min(N_BACKGROUND, len(train)), replace=False)]
        ex = test.iloc[r.choice(len(test), min(N_EXPLAIN, len(test)), replace=False)]
        f = lambda X: m.predict_proba(np.asarray(X))[:, 1]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            sv = shap.explainers.Exact(f, shap.maskers.Independent(bg, max_samples=len(bg)))(ex[H.FEATURES].values,
                                                                                               silent=True)
        parts.append(ex[["BH_Code", "cell", "measured", *H.FEATURES]].assign(
            model=model, fold=k, base=sv.base_values, **{f"shap_{NAMES[c]}": sv.values[:, i] for i, c in enumerate(H.FEATURES)}))
    print(model, "done", flush=True)
shap_df = pd.concat(parts, ignore_index=True)
shap_df.to_csv(PRIV / "shap_values_by_cell.csv", index=False)
glob_ = shap_df.groupby("model")[[f"shap_{v}" for v in NAMES.values()]].apply(lambda d: d.abs().mean()).round(4)
glob_.columns = [c.replace("shap_", "") for c in glob_.columns]
glob_.to_csv(OUT / "shap_mean_abs_by_model.csv")
display(glob_)
print(f"Finished in {(time.time() - T0) / 60:.1f} min. Next: notebook 07.")